In [7]:
import os
import cv2
import glob
import numpy as np
import random
import matplotlib.pyplot as plt
from skimage.transform import resize
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback
import pandas as pd

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Image dimensions (from your context)
IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS = 352, 352, 3

# Dataset paths
TRAIN_IMAGE_PATH = 'Train-Input1'  # jpg
print(len(TRAIN_IMAGE_PATH))
TRAIN_MASK_PATH = 'Train-mask1'    # jpg
TEST_IMAGE_PATH = 'Test-Input'    # jpg
TEST_MASK_PATH = 'Test-mask'      # jpg

# Mount Google Drive


# Custom metrics
def specificity(y_true, y_pred):
    true_negatives = tf.reduce_sum(tf.round(tf.clip_by_value((1 - y_true) * (1 - y_pred), 0, 1)))
    possible_negatives = tf.reduce_sum(tf.round(tf.clip_by_value(1 - y_true, 0, 1)))
    return true_negatives / (possible_negatives + tf.keras.backend.epsilon())

def dice_score(y_true, y_pred):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + tf.keras.backend.epsilon()) / (
        tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + tf.keras.backend.epsilon()
    )

def iou_score(y_true, y_pred):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(tf.clip_by_value(y_pred, 0, 1))
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    union = tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) - intersection
    return (intersection + tf.keras.backend.epsilon()) / (union + tf.keras.backend.epsilon())

# Custom loss: Binary Focal + Dice
def binary_focal_dice_loss(y_true, y_pred):
    # Focal Loss
    alpha = 0.25
    gamma = 2.0
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred, from_logits=False)
    bce_exp = tf.exp(-bce)
    focal_loss = alpha * (1 - bce_exp) ** gamma * bce
    focal_loss = tf.reduce_mean(focal_loss)
    
    # Dice Loss
    intersection = tf.reduce_sum(y_true * y_pred)
    dice_loss = 1 - (2. * intersection + tf.keras.backend.epsilon()) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + tf.keras.backend.epsilon()
    )
    
    return focal_loss + dice_loss

# Custom callback for per-epoch metrics
class MetricsLogger(Callback):
    def __init__(self, validation_data):
        super().__init__()
        self.validation_data = validation_data
        self.metrics_log = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_x, val_y = self.validation_data
        val_pred = self.model.predict(val_x, verbose=0)
        
        val_loss = logs.get('val_loss', 0)
        val_accuracy = logs.get('val_accuracy', 0)
        val_dice = logs.get('val_dice_score', 0)
        val_iou = logs.get('val_iou_score', 0)
        val_precision = logs.get('val_precision', 0)
        val_recall = logs.get('val_recall', 0)
        val_specificity = logs.get('val_specificity', 0)
        
        log_entry = {
            'epoch': epoch + 1,
            'val_loss': val_loss,
            'val_accuracy': val_accuracy,
            'val_dice_score': val_dice,
            'val_iou_score': val_iou,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'val_specificity': val_specificity
        }
        self.metrics_log.append(log_entry)
        
        print(f"\n--- Epoch {epoch + 1} Validation Metrics ---")
        for key, value in log_entry.items():
            if key != 'epoch':
                print(f"{key}: {value:.4f}")
        
        # Save CSV every 10 epochs
        if (epoch + 1) % 10 == 0:
            df = pd.DataFrame(self.metrics_log)
            df.to_csv('metrics_log.csv', index=False)
            print(f"Metrics up to epoch {epoch + 1} saved to metrics_log.csv")

    def on_train_end(self, logs=None):
        df = pd.DataFrame(self.metrics_log)
        df.to_csv('metrics_log.csv', index=False)
        print("Full metrics log saved to Google Drive!")

# ASPP (from your provided code)
def ASPP(inputs):
    shape = inputs.shape
    y_pool = layers.AveragePooling2D(pool_size=(shape[1], shape[2]), name='average_pooling')(inputs)
    y_pool = layers.Conv2D(filters=256, kernel_size=1, padding='same', use_bias=False)(y_pool)
    y_pool = layers.BatchNormalization(name='bn_1')(y_pool)
    y_pool = layers.Activation('relu', name='relu_1')(y_pool)
    y_pool = layers.UpSampling2D((shape[1], shape[2]), interpolation="bilinear")(y_pool)

    y_1 = layers.Conv2D(filters=256, kernel_size=1, dilation_rate=1, padding='same', use_bias=False)(inputs)
    y_1 = layers.BatchNormalization()(y_1)
    y_1 = layers.Activation('relu')(y_1)

    y_6 = layers.Conv2D(filters=256, kernel_size=3, dilation_rate=6, padding='same', use_bias=False)(inputs)
    y_6 = layers.BatchNormalization()(y_6)
    y_6 = layers.Activation('relu')(y_6)

    y_12 = layers.Conv2D(filters=256, kernel_size=3, dilation_rate=12, padding='same', use_bias=False)(inputs)
    y_12 = layers.BatchNormalization()(y_12)
    y_12 = layers.Activation('relu')(y_12)

    y_18 = layers.Conv2D(filters=256, kernel_size=3, dilation_rate=18, padding='same', use_bias=False)(inputs)
    y_18 = layers.BatchNormalization()(y_18)
    y_18 = layers.Activation('relu')(y_18)

    y = layers.Concatenate()([y_pool, y_1, y_6, y_12, y_18])
    y = layers.Conv2D(filters=256, kernel_size=1, dilation_rate=1, padding='same', use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    y = layers.Activation('relu')(y)
    return y

# DeepLabV3+ with ResNet50
def DeepLabV3Plus_resnet50(input_size=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)):
    inputs = layers.Input(input_size)
    base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=inputs)
    
    # Encoder
    image_features = base_model.get_layer('conv4_block6_out').output
    x_a = ASPP(image_features)
    x_a = layers.UpSampling2D((4, 4), interpolation="bilinear")(x_a)
    
    # Low-level features
    x_b = base_model.get_layer('conv2_block2_out').output
    x_b = layers.Conv2D(filters=48, kernel_size=1, padding='same', use_bias=False)(x_b)
    x_b = layers.BatchNormalization()(x_b)
    x_b = layers.Activation('relu')(x_b)
    
    x = layers.Concatenate()([x_a, x_b])
    
    # Decoder
    x = layers.Conv2D(filters=256, kernel_size=3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    x = layers.Conv2D(filters=256, kernel_size=3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.UpSampling2D((4, 4), interpolation="bilinear")(x)
    
    # Output
    x = layers.Conv2D(1, (1, 1), name='output_layer')(x)
    x = layers.Activation('sigmoid')(x)
    
    model = models.Model(inputs, x)
    return model

# Data loading (handles mismatch)
def load_data(image_path, mask_path, target_size=(IMG_HEIGHT, IMG_WIDTH)):
    image_files = sorted(glob.glob(os.path.join(image_path, '*.jpg')))
    mask_files = sorted(glob.glob(os.path.join(mask_path, '*.png')))
    
    min_len = min(len(image_files), len(mask_files))
    image_files = image_files[:min_len]
    mask_files = mask_files[:min_len]
    
    images = np.zeros((min_len, *target_size, IMG_CHANNELS), dtype=np.float32)
    masks = np.zeros((min_len, *target_size, 1), dtype=np.float32)
    
    for i, (img_file, mask_file) in enumerate(zip(image_files, mask_files)):
        img = cv2.imread(img_file)
        if img is None:
            print(f"Warning: Could not load image {img_file}")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = resize(img, target_size, mode='constant', preserve_range=True) / 255.0
        images[i] = img
        
        mask = cv2.imread(mask_file, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            print(f"Warning: Could not load mask {mask_file}")
            continue
        mask = resize(mask, target_size, mode='constant', preserve_range=True)
        mask = np.expand_dims((mask > 127).astype(np.float32), -1)
        masks[i] = mask
    
    print(f"Loaded {len(images)} paired samples from {image_path}")
    return images, masks

# Load data
print("Loading training data...")
Train_Inputs, Train_Masks = load_data(TRAIN_IMAGE_PATH, TRAIN_MASK_PATH)
print("Loading test data...")
Test_Inputs, Test_Masks = load_data(TEST_IMAGE_PATH, TEST_MASK_PATH)

# Split: Exactly 100 for validation
val_size = 100 / len(Train_Inputs)
Train_Inputs, Val_Inputs, Train_Masks, Val_Masks = train_test_split(
    Train_Inputs, Train_Masks, test_size=val_size, random_state=42
)
print(f"Train: {len(Train_Inputs)}, Val: {len(Val_Inputs)}, Test: {len(Test_Inputs)}")

# Create and compile model
model = DeepLabV3Plus_resnet50()
model.compile(
    optimizer=Adam(learning_rate=0.0001),  # Lower LR for transfer learning
    loss=binary_focal_dice_loss,
    metrics=['accuracy', dice_score, specificity, iou_score, 'precision', 'recall']
)
print(model.summary())

# Callbacks
callbacks = [
    ModelCheckpoint('best_model.h5', monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    MetricsLogger((Val_Inputs, Val_Masks))
]

# Train
print("Starting training...")
history = model.fit(
    Train_Inputs, Train_Masks,
    batch_size=8,  # Smaller batch for GPU memory
    epochs=50,     # Reduced epochs for faster training; adjust as needed
    validation_data=(Val_Inputs, Val_Masks),
    callbacks=callbacks,
    verbose=1
)

# Save model
model.save('deeplabv3plus_resnet50.h5')
print("Model saved!")

# Plot history
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(history.history['loss'], label='Train Loss')
axes[0, 0].plot(history.history['val_loss'], label='Val Loss')
axes[0, 0].set_title('Loss')
axes[0, 0].legend()

axes[0, 1].plot(history.history['accuracy'], label='Train Acc')
axes[0, 1].plot(history.history['val_accuracy'], label='Val Acc')
axes[0, 1].set_title('Accuracy')
axes[0, 1].legend()

axes[1, 0].plot(history.history['dice_score'], label='Train Dice')
axes[1, 0].plot(history.history['val_dice_score'], label='Val Dice')
axes[1, 0].set_title('Dice Score')
axes[1, 0].legend()

axes[1, 1].plot(history.history['iou_score'], label='Train IoU')
axes[1, 1].plot(history.history['val_iou_score'], label='Val IoU')
axes[1, 1].set_title('IoU Score')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# Evaluate on test set
print("Evaluating on test set...")
test_results = model.evaluate(Test_Inputs, Test_Masks, verbose=1)
metric_names = ['loss', 'accuracy', 'dice_score', 'specificity', 'iou_score', 'precision', 'recall']
for name, value in zip(metric_names, test_results):
    print(f"Test {name}: {value:.4f}")

# Predict and visualize
preds_test = model.predict(Test_Inputs, verbose=1)
preds_test_t = (preds_test > 0.5).astype(np.uint8)

# Show one random test sample
ix = random.randint(0, len(Test_Masks) - 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(Test_Inputs[ix])
axes[0].set_title('Test Image')
axes[1].imshow(np.squeeze(preds_test_t[ix]), cmap='gray')
axes[1].set_title('Predicted Plaque Mask')
axes[2].imshow(np.squeeze(Test_Masks[ix]), cmap='gray')
axes[2].set_title('Ground Truth Mask')
plt.show()

print("Done! Metrics saved to metrics_log.csv")

12
Loading training data...
Loaded 685 paired samples from Train-Input1
Loading test data...
Loaded 53 paired samples from Test-Input
Train: 585, Val: 100, Test: 53
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 139s 1us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 352, 352,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 358, 358,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 176, 176,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 176, 176,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 176, 176,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 178, 178,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 88, 88,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 88, 88,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 88, 88,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 88, 88,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 88, 88,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 88, 88,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 88, 88,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 88, 88,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 88, 88,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 88, 88,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 88, 88,    │      1,024 │ conv2_block1_3_c

 Total params: 17,830,209 (68.02 MB)

 Trainable params: 17,795,425 (67.88 MB)

...


--- Epoch 1 Validation Metrics ---
val_loss: 1.0044
val_accuracy: 0.9421
val_dice_score: 0.0137
val_iou_score: 0.0069
val_precision: 0.0003
val_recall: 0.0026
val_specificity: 0.9483
74/74 ━━━━━━━━━━━━━━━━━━━━ 698s 9s/step - accuracy: 0.9750 - dice_score: 0.1383 - iou_score: 0.0783 - loss: 0.8723 - precision: 0.1412 - recall: 0.8188 - specificity: 0.9760 - val_accuracy: 0.9421 - val_dice_score: 0.0137 - val_iou_score: 0.0069 - val_loss: 1.0044 - val_precision: 3.2182e-04 - val_recall: 0.0026 - val_specificity: 0.9483 - learning_rate: 1.0000e-04
Epoch 2/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.9952 - dice_score: 0.3713 - iou_score: 0.2362 - loss: 0.6317 - precision: 0.5229 - recall: 0.7825 - specificity: 0.9963
Epoch 2: val_loss improved from 1.00440 to 0.97240, saving model to best_model.h5



--- Epoch 2 Validation Metrics ---
val_loss: 0.9724
val_accuracy: 0.9935
val_dice_score: 0.0299
val_iou_score: 0.0153
val_precision: 0.0000
val_recall: 0.0000
val_specificity: 1.0000
74/74 ━━━━━━━━━━━━━━━━━━━━ 652s 9s/step - accuracy: 0.9961 - dice_score: 0.4126 - iou_score: 0.2703 - loss: 0.5848 - precision: 0.5669 - recall: 0.7817 - specificity: 0.9971 - val_accuracy: 0.9935 - val_dice_score: 0.0299 - val_iou_score: 0.0153 - val_loss: 0.9724 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_specificity: 1.0000 - learning_rate: 1.0000e-04
Epoch 3/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.9963 - dice_score: 0.5242 - iou_score: 0.3667 - loss: 0.4785 - precision: 0.6126 - recall: 0.7797 - specificity: 0.9974
Epoch 3: val_loss did not improve from 0.97240

--- Epoch 3 Validation Metrics ---
val_loss: 0.9775
val_accuracy: 0.9935
val_dice_score: 0.0267
val_iou_score: 0.0135
val_precision: 0.0000
val_recall: 0.0000
val_specificity: 1.0000
74/74 ━━━━━━━━━━━━━━━━━━━━ 661


--- Epoch 4 Validation Metrics ---
val_loss: 0.9644
val_accuracy: 0.9935
val_dice_score: 0.0393
val_iou_score: 0.0201
val_precision: 0.0000
val_recall: 0.0000
val_specificity: 1.0000
74/74 ━━━━━━━━━━━━━━━━━━━━ 665s 9s/step - accuracy: 0.9974 - dice_score: 0.6149 - iou_score: 0.4543 - loss: 0.3801 - precision: 0.7236 - recall: 0.7544 - specificity: 0.9986 - val_accuracy: 0.9935 - val_dice_score: 0.0393 - val_iou_score: 0.0201 - val_loss: 0.9644 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_specificity: 1.0000 - learning_rate: 1.0000e-04
Epoch 5/50
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.9973 - dice_score: 0.6623 - iou_score: 0.5031 - loss: 0.3399 - precision: 0.7170 - recall: 0.7882 - specificity: 0.9984
Epoch 5: val_loss improved from 0.96442 to 0.95924, saving model to best_model.h5



--- Epoch 5 Validation Metrics ---
val_loss: 0.9592
val_accuracy: 0.9935
val_dice_score: 0.0447
val_iou_score: 0.0229
val_precision: 0.0000
val_recall: 0.0000
val_specificity: 1.0000
74/74 ━━━━━━━━━━━━━━━━━━━━ 663s 9s/step - accuracy: 0.9976 - dice_score: 0.6689 - iou_score: 0.5115 - loss: 0.3269 - precision: 0.7426 - recall: 0.7769 - specificity: 0.9987 - val_accuracy: 0.9935 - val_dice_score: 0.0447 - val_iou_score: 0.0229 - val_loss: 0.9592 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_specificity: 1.0000 - learning_rate: 1.0000e-04
Epoch 6/50
 6/74 ━━━━━━━━━━━━━━━━━━━━ 9:31 8s/step - accuracy: 0.9968 - dice_score: 0.6958 - iou_score: 0.5371 - loss: 0.3072 - precision: 0.6350 - recall: 0.8878 - specificity: 0.9973

KeyboardInterrupt: 